# Week 6 Day 1 — Hedging Math

Proof-of-concept for `src/termstructure/risk/hedge.py`.

Goal: for a target bond, compute 3-factor exposures and solve for hedge weights using 2Y/5Y/10Y benchmark instruments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from termstructure.curves.svensson import svensson_zero_rate
from termstructure.risk.hedge import build_hedge, bond_factor_exposures, KRD_BUCKETS

## 1. Load inputs
PCA loadings and a single date of Svensson parameters.

In [ ]:
# PCA loadings — first 3 rows are PC1/PC2/PC3
# Columns: pc, var_ratio, y_1, y_2, y_3, y_5, y_7, y_10, y_20, y_30
loadings_df = pd.read_parquet("../data/processed/pca_loadings.parquet")
loadings = loadings_df.iloc[:3, 2:].to_numpy()  # shape (3, 8)
print("Loadings shape:", loadings.shape)
loadings_df.iloc[:3]

In [ ]:
params = pd.read_parquet("../data/processed/svensson_params.parquet")
params["date"] = pd.to_datetime(params["date"])
row = params.dropna().iloc[-1]
print("Date:", row["date"].date())

# Svensson zero curve at a fine grid for KRD interpolation
fine_grid = np.array([0.5, 1, 2, 3, 5, 7, 10, 15, 20, 25, 30], dtype=float)
curve_rates = np.array([
    svensson_zero_rate(t, row.beta0, row.beta1, row.beta2, row.beta3, row.lambda1, row.lambda2)
    for t in fine_grid
])
print("Curve rates at [1,2,5,10,30]Y:", np.round(curve_rates[[1,2,4,6,10]], 4))

## 2. Factor exposures across the maturity grid
Compute the 3-factor exposure for every maturity in DEFAULT_MATURITIES.

Expected pattern:
- PC1 exposure ≈ modified duration (monotonically larger for longer bonds)
- PC2 exposure changes sign around 5-7Y (slope)
- PC3 is hump-shaped (curvature)

In [ ]:
maturities = [1, 2, 3, 5, 7, 10, 20, 30]
records = []
for mat in maturities:
    coupon = float(svensson_zero_rate(
        mat, row.beta0, row.beta1, row.beta2, row.beta3, row.lambda1, row.lambda2
    ))
    exp = bond_factor_exposures(coupon, float(mat), fine_grid, curve_rates, loadings)
    records.append({"maturity": mat, "PC1": exp[0], "PC2": exp[1], "PC3": exp[2]})

exp_df = pd.DataFrame(records)
exp_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
labels = ["PC1 (Level)", "PC2 (Slope)", "PC3 (Curvature)"]
for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.plot(exp_df["maturity"], exp_df[f"PC{i+1}"], marker="o")
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_title(label)
    ax.set_xlabel("Maturity (years)")
    ax.set_ylabel("Factor exposure (duration units)")
    ax.set_xticks(maturities)
plt.suptitle("Bond factor exposures vs maturity", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Build hedge for a 7Y target bond
Solve 3×3 linear system: find 2Y/5Y/10Y weights that zero out all three factor exposures.

In [ ]:
target_mat = 7.0
target_coupon = float(svensson_zero_rate(
    target_mat, row.beta0, row.beta1, row.beta2, row.beta3, row.lambda1, row.lambda2
))

hedge_mats = [2.0, 5.0, 10.0]
hedge_coupons = [
    float(svensson_zero_rate(m, row.beta0, row.beta1, row.beta2, row.beta3, row.lambda1, row.lambda2))
    for m in hedge_mats
]

result = build_hedge(
    target_coupon, target_mat,
    hedge_coupons, hedge_mats,
    fine_grid, curve_rates,
    loadings,
)

print(f"Target {target_mat}Y — factor exposures:")
for k, label in enumerate(["PC1", "PC2", "PC3"]):
    print(f"  {label}: {result['target_exposures'][k]:.4f}")

print(f"\nHedge weights (short these to hedge the long):")
for m, w in zip(hedge_mats, result['hedge_weights']):
    print(f"  {int(m)}Y: {w:.4f}")

print(f"\nCondition number: {result['condition_number']:.2f}")

In [ ]:
# Residual exposure after hedging — must be ~zero for all 3 factors
residual = result["target_exposures"] + result["hedge_exposures"] @ result["hedge_weights"]
print("Residual factor exposures after hedge:")
for k, label in enumerate(["PC1", "PC2", "PC3"]):
    print(f"  {label}: {residual[k]:.2e}")

## 4. Condition number sweep
Check that the 2Y/5Y/10Y hedge set stays well-conditioned across the maturity grid.
A spike here would mean some target maturity is hard to hedge with these three instruments.

In [ ]:
cond_numbers = []
for mat in maturities:
    coupon = float(svensson_zero_rate(
        mat, row.beta0, row.beta1, row.beta2, row.beta3, row.lambda1, row.lambda2
    ))
    r = build_hedge(coupon, float(mat), hedge_coupons, hedge_mats, fine_grid, curve_rates, loadings)
    cond_numbers.append(r["condition_number"])

plt.figure(figsize=(7, 4))
plt.bar([str(m) for m in maturities], cond_numbers)
plt.axhline(1e4, color="red", linestyle="--", label="1e4 warning threshold")
plt.xlabel("Target maturity (years)")
plt.ylabel("Condition number")
plt.title("Hedge matrix condition number by target maturity")
plt.legend()
plt.tight_layout()
plt.show()

for m, c in zip(maturities, cond_numbers):
    print(f"  {m:2d}Y  cond = {c:.1f}")